In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# It is recommended to use the new seaborn objects interface
import seaborn.objects as so
import os

# fmn settings

In [2]:
# Dynamic output paths system for pipelines
base_output_path = '../out'
project_name = 'merged'
formatted_data_dir = 'formatting'
# Set the type of fmn to use: 'bfmns' or 'tfmns'
# (this is used for the folder name, not for the file names)
fmn_type = 'bfmns'
# Keywords that we want to check for in the textual data. If not empty,
# semantic frame plots will be created for these keywords. Otherwise,
# the full FMNs will be visualized.
keywords = []
keywords = ['matematica', 'ansia', 'stress', 'depressione']
# Lemmatization setting
lemmatize = False

# Types of output: aggregated and individual (entire dataset vs per participant FMNs)
# Choose any subset: {'aggregated'}, {'individual'}, or both
# How to use:
# Only aggregated: methods = {'aggregated'}
# Only individual: methods = {'individual'}
# Both: methods = {'aggregated', 'individual'}
ALLOWED_METHODS = {'aggregated', 'individual'}
methods = {'aggregated'}
methods = {m.strip().lower() for m in (methods if isinstance(methods, (list, set, tuple)) else [methods])}
methods = methods & ALLOWED_METHODS
if not methods:
    raise ValueError("No valid methods selected. Use 'aggregated' and/or 'individual'.")

# Groups to subset the data (if needed). Use labels and .startswith() condition
# from the 'participant_id' column to create groups.
group_name = 'physics'  # fmn name when using groups
if group_name:
    fmn_type = f'{group_name}_{fmn_type}'

# If dfs_and_group_labels is longer than 1, merge the data from the different paths.
dfs_and_group_labels = []
df1 = pd.read_csv('../out/formatting/splitted_dfs/high_anxiety_physics_students_2025.csv')
df2 = pd.read_csv('../out/formatting/splitted_dfs/low_anxiety_physics_students_2025.csv')
dfs_and_group_labels = [
    (df1, None),
    (df2, None)
]

# print all unique participants
all_participants = set()
for df, group_labels in dfs_and_group_labels:
    all_participants.update(df['participant_id'].unique())
print(f"All unique participants across datasets: {len(all_participants)}")
print(f"Unique participants labels: {all_participants}\n")

# Resolve output directories
if methods == {'aggregated'}:
    files_output_dir = f'{base_output_path}/{project_name}/aggregated/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/aggregated/{formatted_data_dir}/{fmn_type}'
if methods == {'individual'}:
    files_output_dir = f'{base_output_path}/{project_name}/individual/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/individual/{formatted_data_dir}/{fmn_type}'
# Set a common plots output directory
plots_output_dir = f'{files_output_dir}/plots'

All unique participants across datasets: 10
Unique participants labels: {'fisica_stem_002', 'fisica_stem_001', 'fisica_015', 'fisica_016', 'fisica_012', 'fisica_005', 'fisica_003', 'fisica_stem_003', 'fisica_010', 'fisica_007'}



# functions

In [3]:
def make_plot_by_type(
    df,
    features_to_plot: list,
    save_path: str,
    fmn_type: str,
    plot_type: str,
    filename=None,
):
    """
    Create plots of type <plot_type> for desired structural features.
    Saves one .png file per feature.

    Parameters:
    df (pd.DataFrame): DataFrame containing the features to plot.
    features_to_plot (list): List of feature names (columns in df) to plot.
    save_path (str): Directory where to save the plots.
    filename (str): optional filename for the plots. None by default.
    fmn_type (str): Type of FMN (e.g., 'BFMN', 'TFMN') for labeling.
    plot_type (str): Type of plot to create.

    Supported plot types (new types soon):
    - 'distribution': Histogram showing the distribution of feature values.
    """
    # Update if new plot types are added
    supported_plot_types = ['distribution']
    
    if plot_type not in supported_plot_types:
        raise ValueError("Type must be one of the following: 'distribution'")

    # Ensure output dirs exist (base + per-plot-type)
    os.makedirs(save_path, exist_ok=True)
    out_dir = os.path.join(save_path, plot_type)
    os.makedirs(out_dir, exist_ok=True)

    for feature in features_to_plot:
        if feature not in df.columns:
            print(f"[WARN] Skipping missing feature: {feature}")
            continue  # skip instead of raising

        feature_series = df[feature].dropna()
        if feature_series.empty:
            print(f"[WARN] Skipping empty feature: {feature}")
            continue  # skip instead of raising

        unique_count = feature_series.nunique()

        # Handle constant-valued feature explicitly (avoids seaborn/numpy bin errors)
        if unique_count == 1:
            const_val = feature_series.iloc[0]
            fig, ax = plt.subplots()
            ax.bar([const_val], [len(feature_series)], width=0.6, alpha=0.6, color="C0")
            ax.set_xlabel(str(feature).replace('_', ' ').title())
            ax.set_ylabel("Count")
            ax.set_title(
                f"Distribution of '{str(feature).replace('_', ' ').title()}' in {fmn_type}s (constant value {const_val}) (n={len(feature_series)})"
            )
            ax.legend([f"Value = {const_val}"])
            if filename:
                out_path = f"{out_dir}/{filename}.png"
            else:
                out_path = f"{out_dir}/{feature}_{plot_type}_{fmn_type}.png"
            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            print(f"[OK] Saved {out_path} (constant feature)")
            continue

        # Heuristic bin width
        if pd.api.types.is_integer_dtype(feature_series):
            bin_width = 1
        else:
            value_range = feature_series.max() - feature_series.min()
            bin_width = value_range / 30 if unique_count > 1 else 1
            if bin_width <= 0 or not np.isfinite(bin_width):
                bin_width = 1

        mean_val = feature_series.mean()
        data = feature_series.to_frame(name=feature)
        
        try:
            if pd.api.types.is_numeric_dtype(feature_series):
                print(f"{feature} is all scalar values")

            plot = (
                so.Plot(data=data, x=feature)
                .add(mark=so.Bars(alpha=0.6), stat=so.Hist(binwidth=bin_width))
                .label(
                    x=feature,
                    y="Count",
                    title=f"Distribution of '{str(feature).replace('_', ' ').title()}' in {fmn_type}s (n={len(feature_series)})"
                )
            )
            fig = plot.plot()
            ax = fig.axes[0]
        except ValueError as e:
            print(f"[INFO] Falling back to seaborn.histplot for {feature}: {e}")
            fig, ax = plt.subplots()
            vmin, vmax = feature_series.min(), feature_series.max()
            if vmin == vmax:
                vmax = vmin + 0.5
            sns.histplot(
                data=data,
                x=feature,
                binwidth=bin_width,
                binrange=(vmin, vmax),
                kde=False,
                alpha=0.6,
                ax=ax
            )
            ax.set_title(f"Distribution of '{str(feature).replace('_', ' ').title()}' in {fmn_type}s (n={len(feature_series)})")
            ax.set_xlabel(str(feature).replace('_', ' ').title())
            ax.set_ylabel("Count")

        ax.axvline(mean_val, color="red", linestyle="--", linewidth=1, label=f"Mean = {mean_val:.2f}")
        ax.legend()
        if filename:
            out_path = f"{out_dir}/{filename}.png"
        else:
            out_path = f"{out_dir}/{feature}_{plot_type}_{fmn_type}.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"[OK] Saved {out_path}\n")

# FMNs structural features to plot

In [4]:
# A list of all the structural features we want to create distributions for (keep empty to plot all the features found in the dataframe)
features = []

# Dynamically establish structural features to plot (all if empty)
if len(features) == 0:
    features = df.columns.tolist()

## TFMNs

In [5]:
# import a fmn structural features dataframe
df = pd.read_csv('../out/merged/aggregated/psychology_low_anxiety_tfmns/structural_features/psychology_low_anxiety_structural_features.csv', index_col=0)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../out/merged/aggregated/psychology_low_anxiety_tfmns/structural_features/psychology_low_anxiety_structural_features.csv'

### Plot

In [ ]:
# call the function for tfmn
make_plot_by_type(
    df=df,
    features_to_plot=features,
    save_path=plots_output_dir,
    fmn_type='tfmn',
    plot_type='distribution',
)

[WARN] Skipping missing feature: participant_id
[WARN] Skipping missing feature: cue_word
[WARN] Skipping missing feature: valence_cue_word
[WARN] Skipping missing feature: association_1
[WARN] Skipping missing feature: valence_association_1
[WARN] Skipping missing feature: association_2
[WARN] Skipping missing feature: valence_association_2
[WARN] Skipping missing feature: association_3
[WARN] Skipping missing feature: valence_association_3


## BFMNs

In [ ]:
# import the features dataframe
df = pd.read_csv('../out/example/path.csv', index_col=0)

FileNotFoundError: [Errno 2] No such file or directory: '../out/example/path.csv'

### Plot

In [ ]:
# call the function for tfmn
make_plot_by_type(
    df=df,
    features_to_plot=features,
    save_path=plots_output_dir,
    fmn_type='bfmn',
    plot_type='distribution',
)